# **Who**: David

## **Task**: find out what phrases/keywords are important for the positively annotated class with respect to the negatives.

Ingredients:
* simply word frequencies: first remove stopwords and extract the word counts (use sklearn's CountVectorizer perhaps), maybe do this per section of the text
* use Jacopo's models in combination with a tool like LIME or SHAP.
* aggregate the embeddings of the medical terms in the texts 

In [1]:
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Scarica le risorse necessarie
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [3]:
df_consults = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

In [ ]:
print(df_consults['text'][4])

### Stopwords removal from the consults dataset

In [4]:
# Carica le stopwords per la lingua olandese
stop_words = stopwords.words('dutch')

# Definisci una funzione per eliminare le stopwords dai testi
def remove_stopwords(testo):
    parole = nltk.word_tokenize(testo)  # Tokenizza il testo
    
    parole = [parola for parola in parole if parola.lower() not in stop_words]  # Rimuove le stopwords
    
    return ' '.join(parole)  # Restituisce il testo senza stopwords

# Applica la funzione al testo
df_consults['clean_text'] = df_consults['text'].apply(remove_stopwords)

In [ ]:
# Funzione per contare le parole in un testo
def conta_parole(testo):
    return len(testo.split())

# Applica la funzione alla colonna 'testo'
print(f'Before stopwords removal: {round(df_consults["text"].apply(conta_parole).mean())}')
print(f'After stopwords removal: {round(df_consults["clean_text"].apply(conta_parole).mean())}')


In [ ]:
# Crea un'istanza di CountVectorizer
vectorizer = CountVectorizer(lowercase=True)

# Trasforma la colonna 'text' in una matrice di conteggio
X = vectorizer.fit_transform(df_consults['clean_text'])

# Ottieni i nomi delle caratteristiche (parole)
feature_names = vectorizer.get_feature_names_out()

# Somma i conteggi per ottenere le parole più comuni
word_counts = X.toarray().sum(axis=0)

# Mostra le parole più comuni
print(word_counts)

# Crea un DataFrame con i conteggi delle parole
word_count_df = pd.DataFrame({'word': feature_names, 'count': word_counts})

# Ordina il DataFrame in base al conteggio delle parole
word_count_df = word_count_df.sort_values(by='count', ascending=False)

# Mostra le parole più comuni
print(word_count_df)